# GPS Sensor

← [f_h Functions](../f_h_functions.ipynb)

Sensor model, argument reference, and ROSNode wiring for the `gps` block.

## Overview

The GPS block is a **stateless** `DynamicalSystem` (no `f`; only `h`).
`h` extracts the position components `[lat, lon, alt]` from the state vector
and adds Gaussian measurement noise, producing an `ndarray(3,)` whose layout
matches `py2ros_nav_sat_fix` exactly.

In a `ROSNode`, the output of `h` is passed directly to `py2ros_nav_sat_fix`,
which converts it to a `sensor_msgs/NavSatFix` message for publication.

## NavSatFix Array Layout

`py2ros_nav_sat_fix` and `ros2py_nav_sat_fix` use the convention:

| Index | Field | Units |
|-------|-------|-------|
| 0 | latitude | degrees |
| 1 | longitude | degrees |
| 2 | altitude | metres (above WGS-84 ellipsoid) |

The `gps_h` output must therefore be a `(3,)` array in this order.

## Noise Model

GPS measurement noise is modelled as zero-mean Gaussian:

$$y_k = \begin{bmatrix} \text{lat} \\ \text{lon} \\ \text{alt} \end{bmatrix} + v_k, \qquad v_k \sim \mathcal{N}(0, R_\text{gps})$$

Typical values for a consumer GPS receiver:

| Component | 1σ accuracy | $R_{ii}$ |
|-----------|------------|----------|
| Latitude | ~3 m ≈ 2.7×10⁻⁵ ° | 7×10⁻¹⁰ |
| Longitude | ~3 m ≈ 3.6×10⁻⁵ ° | 1.3×10⁻⁹ |
| Altitude | ~5 m | 25 |

For a high-precision RTK GPS, shrink these by ~100×.
Off-diagonal entries model correlated errors (usually zero for independent axes).

## Argument Reference

### `xk` — state vector &nbsp; `Float[np.ndarray, "n"]`

Shape `(n,)`. The first 3 elements must be `[lat, lon, alt]`.
Extra state components (velocity, orientation, etc.) are ignored by `gps_h`.

---

### `R_gps` — GPS noise covariance &nbsp; `Float[np.ndarray, "3 3"]`

Shape `(3, 3)`, symmetric positive definite. Encodes GPS measurement uncertainty
in the `[lat, lon, alt]` space. Diagonal entries are the variances of each coordinate;
off-diagonal entries model correlations (typically zero).

```python
R_gps = np.diag([1e-10, 1e-10, 25.0])   # ~1 cm horizontal, 5 m vertical (1σ)
```

This value is typically a fixed parameter — pass it as `static_params` in a deployed `ROSNode`.

## Implementation

In [ ]:
import numpy as np
from jaxtyping import Float, jaxtyped
from beartype import beartype
from sensor_msgs.msg import NavSatFix
from dynamicalnodes import DynamicalSystem
from dynamicalnodes.ros2py_py2ros import py2ros_nav_sat_fix, ros2py_nav_sat_fix

In [ ]:
@jaxtyped(typechecker=beartype)
def gps_h(
    xk: Float[np.ndarray, "n"],
    R_gps: Float[np.ndarray, "3 3"],
) -> Float[np.ndarray, "3"]:
    """GPS sensor model; returns [lat, lon, alt] with simulated measurement noise.

    Args:
        xk:    State vector (n,) — first 3 elements are [lat, lon, alt].
        R_gps: GPS measurement noise covariance (3, 3), units: deg², deg², m².
    """
    noise = np.random.multivariate_normal(np.zeros(3), R_gps)
    return xk[:3] + noise


gps = DynamicalSystem(h=gps_h)

## Worked Example

In [ ]:
xk = np.array([39.5296, -119.8138, 1374.0, 0.0, 0.0])   # lat, lon, alt + extra state
R_gps = np.diag([1e-10, 1e-10, 25.0])                    # ~1 cm horizontal, 5 m vertical

yk = gps.step(xk=xk, R_gps=R_gps)
print("GPS [lat, lon, alt]:", yk)

# py2ros: ndarray(3,) → NavSatFix
msg = py2ros_nav_sat_fix(yk)
print(f"NavSatFix: lat={msg.latitude:.6f}  lon={msg.longitude:.6f}  alt={msg.altitude:.2f} m")

## ROSNode Wiring

Subscribe to a state estimate topic, publish a `NavSatFix`.
The `h` output is a `(3,)` array — `py2ros_nav_sat_fix` converts it to the ROS message.

In [ ]:
from dynamicalnodes import ROSNode
from std_msgs.msg import Float64MultiArray
from dynamicalnodes.ros2py_py2ros import ros2py_float64_multiarray

gps_node = ROSNode(
    dynamical_system=gps,
    subscribes_to=[{
        "topic": "/state_estimate",
        "msg_type": Float64MultiArray,
        "arg": "xk",
        "ros2py": ros2py_float64_multiarray,
        "stale_after": 0.1,
    }],
    publishes_to=[{
        "topic": "/gps/fix",
        "msg_type": NavSatFix,
        "py2ros": py2ros_nav_sat_fix,
    }],
)

# Simulation step
state_msg = Float64MultiArray()
state_msg.data = [39.5296, -119.8138, 1374.0, 0.0, 0.0]
fix = gps_node.step(xk=state_msg, R_gps=R_gps)
print(f"Published NavSatFix: lat={fix.latitude:.6f}  lon={fix.longitude:.6f}")

← [f_h Functions](../f_h_functions.ipynb)